In [2]:
import os
import re
import pandas as pd
import nltk

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger_eng')

data_path = "../data/archive"
output_path = "../data/output"

os.makedirs(output_path, exist_ok=True)

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/nicholasthornton/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/nicholasthornton/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [3]:
kjv = pd.read_csv(os.path.join(data_path, "t_kjv.csv"))
kjv.head()

,id,b,c,v,t
0,1001001,1,1,1,In the beginning God created the heaven and th...
1,1001002,1,1,2,"And the earth was without form, and void; and ..."
2,1001003,1,1,3,"And God said, Let there be light: and there wa..."
3,1001004,1,1,4,"And God saw the light, that it was good: and G..."
4,1001005,1,1,5,"And God called the light Day, and the darkness..."


In [4]:
kjv['token_list'] = kjv['t'].str.lower().apply(
    lambda x: re.findall(r"[a-z']+", x)
)

corpus = (
    kjv
    .explode('token_list')
    .rename(columns={'token_list': 'token_str'})
    .dropna(subset=['token_str'])
    .reset_index(drop=True)
)

corpus['term_str'] = corpus['token_str']

In [5]:
corpus['token_id'] = (
    corpus
    .groupby(['b','c','v'])
    .cumcount() + 1
)

In [6]:
corpus['pos'] = corpus['token_str'].apply(
    lambda x: nltk.pos_tag([x])[0][1]
)

def pos_group(tag):
    if tag.startswith('N'):
        return 'NOUN'
    if tag.startswith('V'):
        return 'VERB'
    if tag.startswith('J'):
        return 'ADJ'
    if tag.startswith('R'):
        return 'ADV'
    return 'OTHER'

corpus['pos_group'] = corpus['pos'].apply(pos_group)

In [7]:
corpus = corpus[['b','c','v','token_id','token_str','term_str','pos','pos_group']]

corpus = corpus.rename(columns={
    'b':'book_number',
    'c':'chapter',
    'v':'verse'
})

corpus.shape

(789668, 8)

In [8]:
corpus.head()

,book_number,chapter,verse,token_id,token_str,term_str,pos,pos_group
0,1,1,1,1,in,in,IN,OTHER
1,1,1,1,2,the,the,DT,OTHER
2,1,1,1,3,beginning,beginning,VBG,VERB
3,1,1,1,4,god,god,NN,NOUN
4,1,1,1,5,created,created,VBN,VERB


In [9]:
delimiter = "|"
print("Delimiter:", delimiter)

num_obs = corpus.shape[0]
print("Number of observations:", num_obs)
print("Within required range (500k–2M):", 500_000 <= num_obs <= 2_000_000)

ohco_structure = "|".join(["book_number","chapter","verse","token_id"])
print("OHCO Structure:", ohco_structure)

columns_string = "|".join(corpus.columns)
print("Columns:", columns_string)

Delimiter: |
Number of observations: 789668
Within required range (500k–2M): True
OHCO Structure: book_number|chapter|verse|token_id
Columns: book_number|chapter|verse|token_id|token_str|term_str|pos|pos_group


In [10]:
corpus.to_csv(os.path.join(output_path, "CORPUS.csv"), index=False, sep="|")

print("CORPUS saved.")
print("Shape:", corpus.shape)

CORPUS saved.
Shape: (789668, 8)
